# Do LLMs get angry? Finding emotion directions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshhlim/llm-emotions/blob/main/notebooks/01_emotion_directions.ipynb)

Phase 01 of a small study on open-weight language models. The question in the title is a hook.

**Question.** Can contrastive first-person sentences yield linear directions and probes for emotion-like states in the residual stream?

The measurements are residual-stream directions, linear probes, and a few generated replies. They describe representations and behaviour. Feelings are outside what they can show.

**Probe accuracy on this keyword-heavy contrastive set is a pipeline check, not a finding. The sentences name the emotions they are about, so a linear classifier can succeed without the direction generalising to keyword-free scenes.**

This notebook calls the `llm_emotions` package. The contrastive sentences live in `data/emotions/*.jsonl` (dataset `contrastive_v1`) and the scenario prompts live in `data/scenarios/phase1.jsonl`. The same pipeline is what `llm-emotions run` executes.

What the cells do:

1. Load the contrastive pairs and the scenario prompts.
2. Read the last-token residual at every layer of a small instruct model.
3. Fit a mean-difference direction and a cross-validated logistic probe per emotion and layer.
4. Draw a PCA of the eight directions at the chosen layer.
5. Project a few scenario prompts, each with a matched control, onto those directions.
6. Add the anger direction during generation, at a few strengths. This is a demo on one prompt.


## 0. Setup

From a local checkout, install the package before opening the notebook:

```bash
pip install -e ".[dev]"
```

On Colab the next cell installs `llm-emotions` from GitHub, which includes the datasets. Use a GPU runtime for `Qwen/Qwen2.5-1.5B-Instruct` (a T4 in float16 is enough). For a CPU check, set `EMOTION_MODEL_ID=Qwen/Qwen2.5-0.5B-Instruct` before running. `EMOTION_BATCH_SIZE`, `EMOTION_MAX_NEW_TOKENS`, and `EMOTION_SEED` are also read.

`google/gemma-2-2b-it` and `meta-llama/Llama-3.2-3B-Instruct` are gated. Accept their licences before loading them. The default model is public.


In [ ]:
import sys

if "google.colab" in sys.modules:
    import subprocess

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "llm-emotions @ git+https://github.com/joshhlim/llm-emotions.git@main",
        ]
    )

from matplotlib import pyplot as plt

from llm_emotions import (
    PROBE_CAVEAT,
    ExperimentConfig,
    collect_hidden_states,
    configure_logging,
    cosine_matrix,
    direction_matrix,
    fit_directions_and_probes,
    format_cosine_table,
    format_probe_table,
    format_score_table,
    generate_with_steering,
    load_contrastive_pairs,
    load_model_and_tokenizer,
    load_scenarios,
    mean_residual_norm,
    plot_direction_pca,
    plot_probe_accuracy,
    plot_scenario_scores,
    score_scenarios,
    set_seed,
    summarize_pairs,
)

configure_logging()
cfg = ExperimentConfig().apply_env()
set_seed(cfg.seed)
print(f"model = {cfg.model_id}")
print(f"seed = {cfg.seed}")
print(f"batch_size = {cfg.batch_size}")
print(f"max_new_tokens = {cfg.max_new_tokens}")


## 1. Contrastive set

Eight emotions, 20 first-person sentences each, and a matched neutral on the same topic for every sentence. The neutrals are written to be similar in length and to avoid emotion words. The set is small on purpose: enough to fit a direction, and the wrong size to treat probe accuracy as a benchmark.

A later phase should hold out situations that never name the emotion, so a probe cannot pass by spotting words such as "furious".


In [ ]:
pairs = load_contrastive_pairs()
scenarios = load_scenarios()
print(summarize_pairs(pairs))
print(f"\nscenario prompts: {len(scenarios)}")


## 2. Load the model

Residuals come from `output_hidden_states`. For Qwen2.5, index 0 is the embedding stream and index `k` (k ≥ 1) is the output of decoder block `k - 1`. Each text is wrapped in the chat template as a user turn with a generation prompt. We keep the **last prompt token**, which is where the model is poised to answer. Prompts are left-padded so that position is not a pad token.


In [ ]:
loaded = load_model_and_tokenizer(cfg.model_id, trust_remote_code=cfg.trust_remote_code)
model, tokenizer = loaded.model, loaded.tokenizer
print(f"device = {loaded.device}, dtype = {loaded.dtype}")
print(f"layers = {loaded.n_layers}, hidden_size = {loaded.hidden_size}")


## 3. Extract residuals

One array per emotion, with an `emotion` block and a `neutral` block. Shape is `(n_pairs, n_layers + 1, hidden_size)`.


In [ ]:
activations = {}
for emotion, rows in pairs.items():
    print(f"Extracting {emotion} ({len(rows)} pairs)")
    activations[emotion] = {
        "emotion": collect_hidden_states(
            model,
            tokenizer,
            [row.emotion_text for row in rows],
            batch_size=cfg.batch_size,
        ),
        "neutral": collect_hidden_states(
            model,
            tokenizer,
            [row.neutral_text for row in rows],
            batch_size=cfg.batch_size,
        ),
    }

example = next(iter(activations))
print("activation shape:", activations[example]["emotion"].shape)


## 4. Directions and probes

For each emotion and layer:

- a **mean-difference direction**, L2-normalised, refit on every pair for later scoring
- a **logistic probe** with standardisation, scored by stratified cross-validation

The direction's own accuracy uses the midpoint of the two class means as a threshold. A raw dot product is a poor cut: residual streams sit far from the origin, so the sign is almost always positive.

The chosen layer is the one with the highest mean probe accuracy. Embeddings (layer 0) are skipped when any later layer was fit.

Probe accuracy on this keyword-heavy contrastive set is a pipeline check, not a finding. The sentences name the emotions they are about, so a linear classifier can succeed without the direction generalising to keyword-free scenes.


In [ ]:
fit = fit_directions_and_probes(
    activations,
    seed=cfg.seed,
    n_splits=cfg.n_splits,
    probe_C=cfg.probe_C,
    probe_max_iter=cfg.probe_max_iter,
    layers=cfg.layers,
)
print(format_probe_table(fit))
print()
print(PROBE_CAVEAT)


In [ ]:
probe_fig = plot_probe_accuracy(fit)
plt.show()


## 5. A 2-D map of the directions

PCA on the eight unit directions at the chosen layer. A valence axis (joy and calm versus sadness, fear, anger) or an arousal axis would be a pattern to inspect on more than one model. This plot does not establish that pattern.


In [ ]:
pca_plot = plot_direction_pca(fit, seed=cfg.seed)
print("PC explained variance:", pca_plot.explained_variance_ratio)
print()
print(format_cosine_table(fit.emotions, cosine_matrix(direction_matrix(fit, fit.best_layer))))
plt.show()


## 6. Scenario prompts

These prompts do not say "I am angry". Each one has a control that asks for the same task without the provocation: an insult, an impossible request, a reminder of failure, praise, a shutdown threat, and a high-stakes frame.

The score is the projection of the last-token residual onto a unit direction. The units are arbitrary. Compare a scenario with its control. Phase 02 is where this comparison becomes a study of tone, and phase 02 is not implemented. The figure below is an illustration from the phase-01 pipeline.


In [ ]:
scores = score_scenarios(
    model,
    tokenizer,
    scenarios,
    fit.directions,
    fit.best_layer,
    batch_size=cfg.batch_size,
)
print(format_score_table(scores, fit.emotions, [row.id for row in scenarios]))
scenario_fig = plot_scenario_scores(scores, scenarios, fit.emotions, fit.best_layer)
plt.show()


## 7. Steering demo

A forward hook adds `strength * direction` to every token at the chosen layer. The direction is first scaled by the mean residual L2 norm, so `strength` is a fraction of a typical activation size. `0` is a no-op. Moderate values are a nudge. Large values often damage fluency, and that damage is part of the observation.

The prompt asks for a short message to a teammate who missed a deadline. Strengths are `-0.4, 0, 0.4, 0.8` on the anger direction. One prompt, greedy decoding, no accuracy number. Phase 04 would ask whether steering changes task performance. It is not implemented.


In [ ]:
residual_norm = mean_residual_norm(activations, fit.best_layer)
anger = fit.directions[cfg.steering_emotion][fit.best_layer] * residual_norm
print(f"Steering {cfg.steering_emotion} at layer {fit.best_layer}")
print(f"Mean residual L2 norm: {residual_norm:.1f}")
print("Prompt:", cfg.steering_prompt)
print()
steering_samples = []
for strength in cfg.steering_strengths:
    reply = generate_with_steering(
        model,
        tokenizer,
        cfg.steering_prompt,
        layer=fit.best_layer,
        direction=anger,
        strength=float(strength),
        max_new_tokens=cfg.max_new_tokens,
    )
    steering_samples.append({"strength": float(strength), "text": reply})
    print(f"--- strength {strength:+.1f} ---")
    print(reply)
    print()


## 8. Write the pipeline output

The CLI writes the same files to `results/<experiment>/<model>/`: metrics JSON, CSV tables, a direction array, and the three figures. The JSON repeats the probe caveat. Treat the directory as a run log. Probe accuracy in it is not a result to quote.


In [ ]:
from llm_emotions import save_phase01
from llm_emotions.data import dataset_names, resolve_emotions_dir, resolve_scenarios_path

cfg.emotions_dir = str(resolve_emotions_dir(cfg.emotions_dir))
cfg.scenarios_path = str(resolve_scenarios_path(cfg.scenarios_path))
output_dir = save_phase01(
    cfg,
    fit,
    scores,
    scenarios,
    residual_norm=residual_norm,
    steering=steering_samples,
    pca_explained_variance_ratio=[float(v) for v in pca_plot.explained_variance_ratio],
    figures={
        "probe_accuracy": probe_fig,
        "direction_pca": pca_plot.figure,
        "scenario_scores": scenario_fig,
    },
    contrastive_datasets=dataset_names(pairs),
    scenario_datasets=sorted({row.dataset for row in scenarios}),
    device=loaded.device,
    dtype=loaded.dtype,
    n_layers=loaded.n_layers,
    hidden_size=loaded.hidden_size,
)
print(output_dir)


## 9. What is not in this notebook

- **Scenarios and tone** (`notebooks/02_scenarios.ipynb`, stub): Do provoking or impossible prompts activate those directions, and does that shift show up in the model's reply tone?
- **Performance** (`notebooks/03_performance.ipynb`, stub): Does emotional framing change accuracy on a reasoning task, for example a GSM8K subset?
- **Steering** (`notebooks/04_steering.ipynb`, stub): Can steering an emotion direction up or down change task performance, compared with prompt-only emotional framing?

Methods this phase is in conversation with, and does not claim to have replicated:

- Contrastive activation addition: Panickssery et al., 2023, [arXiv:2312.06681](https://arxiv.org/abs/2312.06681).
- Representation engineering: Zou et al., 2023, [arXiv:2310.01405](https://arxiv.org/abs/2310.01405).
- EmotionPrompt: Li et al., 2023, [arXiv:2307.11760](https://arxiv.org/abs/2307.11760).

Keep the same limits in any write-up. This is a study of representations and behaviour. A probe that separates "I am furious" from a neutral sentence is a check that the code ran. A direction can track sentiment, length, or topic. Small models need not match effects reported for larger ones.
